In [200]:
import math
import numpy as np
import random

In [ ]:
class Value:
    def __init__(self, data, _children = (), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self._backward = lambda: None
        self.grad = 0

    def __repr__(self):
        return f"Value(data={self.data}, label={self.label}, gradient={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, _children = (self, other), _op='+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, _children = (self, other), _op='*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out;

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data**other.data, _children = (self, ), _op = f'**{other.data}')
        def _backward():
            d = other.data * (self.data ** (other.data - 1))
            self.grad += d * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return -1 * self

    def __sub__(self, other):
        return self + (-other)

    def __truediv__(self, other):
        return self * (other ** -1)


    def tanh(self):
        _dataoutput = (math.exp(2 * self.data) - 1)/(math.exp(2*self.data) + 1)
        out = Value(_dataoutput, _children= (self, ), _op = 'tanh')
        def _backward():
            t = 1 - (_dataoutput)**2
            self.grad += t * out.grad
        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        visited = set()
        res = []
        def build_topo(node):
            visited.add(node)  
            for child in node._prev:
                if child not in visited:
                    build_topo(child)
            res.append(node)         

        build_topo(self)
        for v in reversed(res):
            v._backward()



In [202]:
a = Value(2, label = 'a')
b = Value(-3, label = 'b')
c = Value(10, label = 'c')
f = Value(-2, label = 'f')
e = a * b
e.label = 'e'
d = e + c
d.label = 'd'
L = d * f
L.label = 'L'
L.backward()


Value(data=2, label=a, gradient=6)
Value(data=-3, label=b, gradient=-4)
Value(data=-6, label=e, gradient=-2)
Value(data=10, label=c, gradient=-2)
Value(data=4, label=d, gradient=-2)
Value(data=-2, label=f, gradient=4)
Value(data=-8, label=L, gradient=1)


In [203]:
a = Value(3.0)
b = a + a
b.backward()

Value(data=3.0, label=, gradient=2)
Value(data=6.0, label=, gradient=1)


In [204]:
x1 = Value(2, label = 'x1')
x2 = Value(0, label = 'x2')
w1 = Value(-3, label = 'w1')
w2 = Value(1, label = 'w2')
b = Value(6.8813735870195432, label = 'b')
x1w1 = x1 * w1
x1w1.label = 'x1*w1'
x2w2 = x2 * w2
x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = 'x1*w1 + x2*w2'
n =  x1w1 + x2w2 + b
n.label = 'x1*w1 + x2*w2 + b'
o = n.tanh()
o.backward()



Value(data=6.881373587019543, label=b, gradient=0.4999999999999999)
Value(data=0, label=x2, gradient=0.4999999999999999)
Value(data=1, label=w2, gradient=0.0)
Value(data=0, label=x2*w2, gradient=0.4999999999999999)
Value(data=-3, label=w1, gradient=0.9999999999999998)
Value(data=2, label=x1, gradient=-1.4999999999999996)
Value(data=-6, label=x1*w1, gradient=0.4999999999999999)
Value(data=-6, label=, gradient=0.4999999999999999)
Value(data=0.8813735870195432, label=x1*w1 + x2*w2 + b, gradient=0.4999999999999999)
Value(data=0.7071067811865476, label=, gradient=1)


In [205]:
a = Value(2, label = 'a')
b = Value(-3, label = 'b')
y = (a-b)/(a**2)
y.label = 'y'
y.backward()


Value(data=2, label=a, gradient=-1.0)
Value(data=4, label=, gradient=-0.3125)
Value(data=0.25, label=, gradient=5)
Value(data=-3, label=b, gradient=-0.25)
Value(data=-1, label=, gradient=-0.75)
Value(data=3, label=, gradient=0.25)
Value(data=5, label=, gradient=0.25)
Value(data=1.25, label=y, gradient=1)


In [206]:
a.data += 0.0001
y = (a-b)/(a**2)
y.backward()

Value(data=2.0001, label=a, gradient=-1.9998625131239374)
Value(data=4.000400010000001, label=, gradient=-0.31244375656187495)
Value(data=0.24997500187487495, label=, gradient=5.0001)
Value(data=-3, label=b, gradient=-0.499975001874875)
Value(data=-1, label=, gradient=-0.7499250056246248)
Value(data=3, label=, gradient=0.24997500187487495)
Value(data=5.0001, label=, gradient=0.24997500187487495)
Value(data=1.2499000068745623, label=, gradient=1)


In [207]:
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, vals):
        dot_products = [v*w for v,w in zip(vals, self.w)]
        res = sum(dot_products, self.b)
        return res.tanh()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, vals):
        outputs = []
        for n in self.neurons:
            outputs.append(n(vals))
        return outputs

    def parameters(self):
        params = []
        for n in self.neurons:
            params += n.parameters();
        return params
            

        

class MLP:
    def __init__(self, nin, nout):
        self.mlp = [nin] + nout
        self.layers = [Layer(self.mlp[i], self.mlp[i+1]) for i in range(len(self.mlp)-1)]
    def __call__(self, inputs):
        for layer in self.layers:
            new_inputs = layer(inputs)
            inputs = new_inputs
        return inputs[0] if len(inputs)==1 else inputs
    def parameters(self):
        params = []
        for layer in self.layers:
            params+= layer.parameters()
        return params

    


In [222]:
xs = [[3.0, -1.0, 2.0], [5.0, 2.0, 3.0], [-1.0, -2.0, -3.0], [6.9, 3.2, 0.0]]
ys = [1, -1, -1, 1]
n = MLP(3, [4, 4, 1])
for k in range(50):
    yout = [n(x) for x in xs];
    loss = sum([(y2-y1) ** 2 for y2,y1 in zip(yout, ys)])
    for p in n.parameters():
        p.grad = 0
    loss.backward()
    for p in n.parameters():
        p.data -= 0.05 * p.grad
yout





Value(data=-1, label=, gradient=-2.5453911408090493)
Value(data=0.08476656695971219, label=, gradient=0.06728946842428522)
Value(data=0.0, label=, gradient=-0.0026624400804886607)
Value(data=0.8656672800899115, label=, gradient=-0.013513387267476692)
Value(data=0.0, label=, gradient=-0.00307559283078382)
Value(data=3.2, label=, gradient=0.0003456028093142161)
Value(data=-0.11236949372981164, label=, gradient=-0.0027427593617660997)
Value(data=-0.35958237993539727, label=, gradient=-0.00307559283078382)
Value(data=0.6918713137840118, label=, gradient=-0.041426177354164345)
Value(data=6.9, label=, gradient=-0.0021279144524990893)
Value(data=4.773912065109681, label=, gradient=-0.00307559283078382)
Value(data=-0.681231516132695, label=, gradient=-0.00987292029543631)
Value(data=4.092680548976986, label=, gradient=-0.00307559283078382)
Value(data=3.7330981690415888, label=, gradient=-0.00307559283078382)
Value(data=3.7330981690415888, label=, gradient=-0.00307559283078382)
Value(data=0.998

[Value(data=0.8681029402032209, label=, gradient=-0.26379411959355825),
 Value(data=0.009582016894330261, label=, gradient=2.0191640337886607),
 Value(data=-0.949146253994216, label=, gradient=0.10170749201156792),
 Value(data=0.010816395126380988, label=, gradient=-1.978367209747238)]